In [2]:
import pandas as pd
# Read the CSV file into a DataFrame
df = pd.read_csv('webnlg_qa_selected_es_v4_corrected.csv')
# Print the DataFrame
print(df.head())
df.to_excel('webnlg_qa_selected_es_v4_corrected.xlsx', index=False)

   split category   eid  lex_id  \
0    dev  Airport   Id1       0   
1  train  Airport   Id1       0   
2    dev  Airport   Id1       0   
3  train  Airport   Id1       1   
4    dev  Airport  Id10       2   

                                            lex_text  \
0          The leader of Aarhus is Jacob Bundsgaard.   
1      The Aarhus is the airport of Aarhus, Denmark.   
2          The leader of Aarhus is Jacob Bundsgaard.   
3  Aarhus Airport serves the city of Aarhus, Denm...   
4  The elevation above the sea level (in metres) ...   

                                         lex_text_es  \
0            El líder de Aarhus es Jacob Bundsgaard.   
1      Aarhus es el aeropuerto de Aarhus, Dinamarca.   
2            El líder de Aarhus es Jacob Bundsgaard.   
3  El aeropuerto de Aarhus da servicio a la ciuda...   
4  La altitud sobre el nivel del mar (en metros) ...   

                   xml_file  \
0  Airport_allSolutions.xml   
1  Airport_allSolutions.xml   
2  Airport_allSolution

In [3]:
# pip install pandas lingua-language-detector

import re
import pandas as pd
from lingua import Language, LanguageDetectorBuilder

INPUT_FILE = "./dataset/ir_texts.csv"
OUTPUT_FILE = "language_check_results.csv"
MISMATCH_FILE = "language_mismatches.csv"

df = pd.read_csv(INPUT_FILE)

# Your file uses these labels
LABEL_TO_LANGUAGE = {
    "en": Language.ENGLISH,
    "es": Language.SPANISH,
}

LANGUAGE_TO_LABEL = {
    Language.ENGLISH: "en",
    Language.SPANISH: "es",
}

detector = (
    LanguageDetectorBuilder
    .from_languages(Language.ENGLISH, Language.SPANISH)
    .with_preloaded_language_models()
    .build()
)

def detect_lang_and_confidence(text):
    if pd.isna(text) or not str(text).strip():
        return pd.Series({
            "detected_lang": None,
            "confidence": 0.0,
            "margin": 0.0,
        })

    text = str(text)

    confidence_values = detector.compute_language_confidence_values(text)
    confidence_values = sorted(
        confidence_values,
        key=lambda x: x.value,
        reverse=True
    )

    top = confidence_values[0]
    second = confidence_values[1] if len(confidence_values) > 1 else None

    detected_label = LANGUAGE_TO_LABEL.get(top.language)
    confidence = top.value
    margin = top.value - second.value if second else top.value

    return pd.Series({
        "detected_lang": detected_label,
        "confidence": confidence,
        "margin": margin,
    })

# Detect language from text
df[["detected_lang", "confidence", "margin"]] = df["text"].apply(detect_lang_and_confidence)

# Main check: does detected language match the lang column?
df["lang_matches_text"] = df["lang"] == df["detected_lang"]

# Optional extra check: does doc_id contain the same language label?
def extract_lang_from_doc_id(doc_id):
    match = re.search(r"__(en|es)__", str(doc_id))
    return match.group(1) if match else None

df["lang_from_doc_id"] = df["doc_id"].apply(extract_lang_from_doc_id)
df["lang_matches_doc_id"] = df["lang"] == df["lang_from_doc_id"]

# Flag rows that should be manually reviewed.
# Low margins often happen with short texts, names, or texts with many proper nouns.
df["needs_review"] = (
    (df["lang_matches_text"] == False) |
    (df["lang_matches_doc_id"] == False) |
    (df["margin"] < 0.15)
)

# Save full results
df.to_csv(OUTPUT_FILE, index=False)

# Save only problematic / uncertain rows
mismatches = df[df["needs_review"]].copy()
mismatches.to_csv(MISMATCH_FILE, index=False)

print("Total rows:", len(df))
print("Text/lang mismatches:", (~df["lang_matches_text"]).sum())
print("doc_id/lang mismatches:", (~df["lang_matches_doc_id"]).sum())
print("Rows needing review:", df["needs_review"].sum())

print("\nSaved:")
print(f"- {OUTPUT_FILE}")
print(f"- {MISMATCH_FILE}")

print("\nSample rows needing review:")
print(
    mismatches[
        ["doc_id", "lang", "detected_lang", "confidence", "margin", "text"]
    ].head(20)
)

Total rows: 33286
Text/lang mismatches: 953
doc_id/lang mismatches: 0
Rows needing review: 1227

Saved:
- language_check_results.csv
- language_mismatches.csv

Sample rows needing review:
                                           doc_id lang detected_lang  \
27    text__train__Airport__Id14__1triples__es__1   es            es   
51    text__train__Airport__Id26__1triples__es__2   es            en   
64    text__train__Airport__Id33__1triples__en__0   en            es   
66    text__train__Airport__Id34__1triples__en__1   en            es   
68    text__train__Airport__Id35__1triples__en__2   en            es   
78    text__train__Airport__Id40__1triples__en__0   en            en   
126   text__train__Airport__Id64__1triples__en__2   en            es   
144   text__train__Airport__Id73__1triples__en__1   en            en   
151   text__train__Airport__Id76__1triples__es__0   es            en   
225  text__train__Airport__Id113__1triples__es__2   es            en   
229  text__train__Ai